# Transform & Deseason

Runs on a **master Excel** workbook; default **`Master_Data_Base_version_Brownian_by_frequency.xlsx`** (from the Brownian-by-frequency notebook). Override with env **`TRANSFORM_DESEASON_INPUT`** or edit **`INPUT_XLSX`** in the first code cell.

**Outputs** are named **`Transformed_Deseason_*__from_<input_stem>.xlsx`** so each run records which master file was used.

**Calendar:** after load, rows are restricted to **business days** (Monday–Friday); weekend dates from calendar-daily masters are dropped.

**Sample window:** features and **C5** use all rows from **2013-01-01** through the **last date in the file** (`df.index.max()`), so **2025+** is included when present. This feeds both the FFT/ACF pipeline and **Part 2** (first-difference).


In [1]:
import os
import pandas as pd
from pathlib import Path

# Path setup: works locally; for Colab, set USE_COLAB=True and run drive mount first
BASE_DIR = os.getcwd()
USE_COLAB = False

if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive/')
    BASE_DIR = '/content/drive/MyDrive/Project 005 Data'

# Default: Brownian-filled master; override with env TRANSFORM_DESEASON_INPUT or edit below
INPUT_XLSX = os.environ.get(
    "TRANSFORM_DESEASON_INPUT", "Master_Data_Base_version_Brownian_by_frequency.xlsx"
)
INPUT_FILE = os.path.join(BASE_DIR, INPUT_XLSX)
if not os.path.exists(INPUT_FILE):
    for d in [os.path.expanduser("~/Downloads/Project 005 Data"), os.path.dirname(os.path.abspath("."))]:
        p = os.path.join(d, INPUT_XLSX)
        if os.path.exists(p):
            BASE_DIR = d
            INPUT_FILE = p
            break

# Optional legacy fallback if Brownian workbook is missing
if not os.path.exists(INPUT_FILE) and INPUT_XLSX.endswith("Brownian_by_frequency.xlsx"):
    _legacy = os.path.join(BASE_DIR, "Master_Data_AVG_version_update.xlsx")
    if os.path.exists(_legacy):
        print(
            "Warning: " + INPUT_XLSX + " not found; using Master_Data_AVG_version_update.xlsx"
        )
        INPUT_FILE = _legacy

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Master workbook not found. Expected: {INPUT_XLSX} under {BASE_DIR}"
    )

INPUT_STEM = Path(INPUT_FILE).stem
print(f"Using: {INPUT_FILE}")
print(f"Output files will include: __from_{INPUT_STEM}")

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Using: /Users/andyyang/Downloads/Project 005 Data/Master_Data_Base_version_Brownian_by_frequency.xlsx
Output files will include: __from_Master_Data_Base_version_Brownian_by_frequency


In [2]:
os.chdir(BASE_DIR)

In [3]:
df = pd.read_excel(INPUT_FILE)


In [4]:
# Ensure Date index; keep only business days (Mon–Fri)
date_col = "Date" if "Date" in df.columns else df.columns[0]
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=[date_col])
df = df.set_index(date_col)
_ix = pd.to_datetime(df.index, errors="coerce")
df = df.loc[_ix.notna() & (_ix.weekday < 5)]
print(f"Rows (business days only): {len(df)}")

Rows (business days only): 5265


In [5]:
features = df.iloc[:, 2:]

# Exclude model-generated variables (e.g. XGBoost predictions) to avoid feature leakage/circular dependency
MODEL_GENERATED_PATTERNS = ["XGBoost_pred", "_pred"]  # extend as needed for other model outputs
exclude_cols = [c for c in features.columns if any(p in c for p in MODEL_GENERATED_PATTERNS)]
if exclude_cols:
    print(f"Excluding model-generated columns: {exclude_cols}")
    features = features.drop(columns=exclude_cols)

In [6]:
# Full master history (includes 2025+ when in the xlsx); used for Part 1 (FFT/ACF) and Part 2 (first difference)
DATA_START = pd.Timestamp("2013-01-01")
DATA_END = df.index.max()
features = features.loc[(features.index >= DATA_START) & (features.index <= DATA_END)]
features


,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes","Japan BFI Production_,000 tonnes","R.O.Korea BFI Production_,000 tonnes",Japan Steel Ship Plate Commodity Price_$/Tonne,...,steel margin cash profit with cost delayed 1 month for marginal producer of Rebar,steel margin cash profit with cost delayed 1 week for marginal producer of Rebar,steel margin cash profit with cost delayed 1 month for marginal producer of HRC,steel margin cash profit with cost delayed 1 week for marginal producer of HRC,steel margin cash profit with cost delayed 1 month for marginal producer of CRC,steel margin cash profit with cost delayed 1 week for marginal producer of CRC,Guinea Total Bauxite Exports,Guinea Bauxite Exports in Capes,CHN Imports from Guinea,CHN Imports from Guinea inCapes
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,9.900000,-5.800000,7.300000,2833.218719,347.498651,243.250273,2357.945485,310.292172,166.114068,630.000000,...,191.733858,4.084625,327.631294,139.982061,-308.266142,-495.915375,1.392190e+06,NaN,NaN,NaN
2013-01-02,9.119526,-6.487948,8.412519,2744.907012,362.292045,268.694052,2850.813524,324.390227,140.651569,570.978642,...,184.809016,3.818757,329.547353,163.599847,-307.513785,-533.209184,1.439254e+06,NaN,NaN,NaN
2013-01-03,8.743418,-5.392167,7.667353,2926.263479,395.185562,267.196156,2584.302096,304.578515,139.894902,598.987014,...,243.722310,3.731504,380.152759,157.908736,-304.596908,-552.897797,1.476722e+06,NaN,NaN,NaN
2013-01-04,9.915606,-4.147681,6.796589,3372.646099,432.950967,258.746924,2668.763126,302.026152,162.541402,662.696328,...,198.733663,3.467400,342.110996,118.959665,-307.779353,-544.794490,1.446866e+06,NaN,NaN,NaN
2013-01-07,9.392962,-5.302108,6.868981,2858.098888,333.948645,240.636157,2113.238323,256.922976,151.745819,711.877334,...,192.894466,4.046496,305.301519,137.775680,-421.108866,-527.529655,1.344928e+06,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-119.020319,-77.657759,-232.273155,-142.868783,-268.036101,-214.724667,NaN,NaN,NaN,NaN
2026-02-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-119.136975,-69.406925,-201.418406,-144.413001,-252.377347,-189.939481,NaN,NaN,NaN,NaN
2026-02-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-109.467805,-88.276860,-193.825438,-177.457345,-298.658380,-222.799734,NaN,NaN,NaN,NaN


In [7]:
C5 = df.C5.loc[(df.index >= DATA_START) & (df.index <= DATA_END)]
C5


Date
2013-01-01      NaN
2013-01-02    6.950
2013-01-03    7.059
2013-01-04    7.109
2013-01-07    7.236
              ...  
2026-02-09      NaN
2026-02-10      NaN
2026-02-11      NaN
2026-02-12      NaN
2026-02-13      NaN
Name: C5, Length: 3424, dtype: float64

# Skewness Analysis on Independent Variable


In [8]:
features_des = features.agg(['mean', 'median', 'std', 'skew', 'kurtosis']).T
features_des.columns = ["Mean", "Median", "STD", "Skewness", "Kurtosis"]
features_des["Direction"] = "Normal"
features_des.loc[features_des["Skewness"] < -0.5, "Direction"] = "Left Skewed"
features_des.loc[features_des["Skewness"] > 0.5, "Direction"] = "Right Skewed"
features_des

,Mean,Median,STD,Skewness,Kurtosis,Direction
Industrial Production China_% Yr/Yr,6.261633e+00,6.056760e+00,4.444178e+00,2.318146,25.089737,Right Skewed
Industrial Production Japan_% Yr/Yr,-1.439091e-01,-6.154589e-01,5.836940e+00,0.017834,5.193966,Normal
Industrial Production S Korea_% Yr/Yr,9.630932e-01,8.161013e-01,4.867229e+00,0.169272,0.563881,Normal
"China Steel Production_,000 tonnes",3.552380e+03,3.501365e+03,5.875440e+02,0.393649,-0.225935,Normal
"Japan Steel Production_,000 tonnes",3.720117e+02,3.717850e+02,5.757104e+01,0.105250,-0.195519,Normal
...,...,...,...,...,...,...
steel margin cash profit with cost delayed 1 week for marginal producer of CRC,-7.302319e+01,-7.317119e+01,4.827275e+02,0.493599,0.348114,Normal
Guinea Total Bauxite Exports,6.377012e+06,5.776992e+06,4.451458e+06,0.817509,0.093675,Right Skewed
Guinea Bauxite Exports in Capes,6.246912e+06,5.372881e+06,4.272930e+06,0.687824,-0.097655,Right Skewed
CHN Imports from Guinea,5.364207e+06,4.498546e+06,4.278854e+06,0.816043,0.107978,Right Skewed


In [9]:
from statsmodels.tsa.stattools import adfuller
station_results = []

for col in features.columns:
    series = features[col].dropna()
    if len(series) < 2 or series.nunique() < 2:
        station_results.append({"Feature": col, "ADF_Statistic": None, "p_value": None, "Stationary": "Skip"})
        continue
    try:
        result = adfuller(series)
        station = "Yes" if result[1] <= 0.05 else "No"
        station_results.append({"Feature": col, "ADF_Statistic": round(result[0], 4), "p_value": round(result[1], 4), "Stationary": station})
    except Exception:
        station_results.append({"Feature": col, "ADF_Statistic": None, "p_value": None, "Stationary": "Skip"})

In [10]:
station_results_df = pd.DataFrame(station_results)
station_results_df.loc[station_results_df["Stationary"] == "No"]
station_results_df.head(10)

,Feature,ADF_Statistic,p_value,Stationary
0,Industrial Production China_% Yr/Yr,-6.2429,0.0000,Yes
1,Industrial Production Japan_% Yr/Yr,-4.1000,0.0010,Yes
2,Industrial Production S Korea_% Yr/Yr,-4.9058,0.0000,Yes
3,"China Steel Production_,000 tonnes",-3.4095,0.0106,Yes
4,"Japan Steel Production_,000 tonnes",-3.0710,0.0288,Yes
5,"South Korea Steel Production_,000 tonnes",-9.0206,0.0000,Yes
6,"P.R. China BFI Production_,000 tonnes",-3.4922,0.0082,Yes
7,"Japan BFI Production_,000 tonnes",-1.9603,0.3042,No
8,"R.O.Korea BFI Production_,000 tonnes",-1.9121,0.3264,No
9,Japan Steel Ship Plate Commodity Price_$/Tonne,-1.2694,0.6430,No


# Transformation Using Box-Cox and Yeo-Johnson

In [11]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method = "yeo-johnson", standardize=True)
pt2 = PowerTransformer(method = "box-cox", standardize=True)

transform_feature = features.copy()

for column in transform_feature.columns:
  if features_des.loc[column, "Direction"] == "Left Skewed":
    transform_feature[column] = pt.fit_transform(features[column].values.reshape(-1, 1))
  elif features_des.loc[column, "Direction"] == "Right Skewed":
    try:
      transform_feature[column] = pt2.fit_transform(features[column].values.reshape(-1, 1))
    except:
      print(column)
      transform_feature[column] = pt.fit_transform(features[column].values.reshape(-1, 1))
  else:
    transform_feature[column] = features[column]

transform_feature

Industrial Production  China_% Yr/Yr
Capesize Bulkcarrier Deliveries_No
Capesize Bulkcarrier Deliveries_DWT
Capesize Fleet Growth_% Yr/Yr
Steel Products Production, Year-to-Date (YTD) Growth (%)
china new loan
EAF Margin Spot vs. Spot(RHS)
 EAF Margin 1 week scrap inventory(RHS)
EAF cash margin (Rmb/t)
steel margin cash profit with cost delayed 1 month for marginal producer of Rebar
steel margin cash profit with cost delayed 1 week for marginal producer of Rebar
steel margin cash profit with cost delayed 1 month for marginal producer of HRC
steel margin cash profit with cost delayed 1 week for marginal producer of HRC


,Industrial Production China_% Yr/Yr,Industrial Production Japan_% Yr/Yr,Industrial Production S Korea_% Yr/Yr,"China Steel Production_,000 tonnes","Japan Steel Production_,000 tonnes","South Korea Steel Production_,000 tonnes","P.R. China BFI Production_,000 tonnes","Japan BFI Production_,000 tonnes","R.O.Korea BFI Production_,000 tonnes",Japan Steel Ship Plate Commodity Price_$/Tonne,...,steel margin cash profit with cost delayed 1 month for marginal producer of Rebar,steel margin cash profit with cost delayed 1 week for marginal producer of Rebar,steel margin cash profit with cost delayed 1 month for marginal producer of HRC,steel margin cash profit with cost delayed 1 week for marginal producer of HRC,steel margin cash profit with cost delayed 1 month for marginal producer of CRC,steel margin cash profit with cost delayed 1 week for marginal producer of CRC,Guinea Total Bauxite Exports,Guinea Bauxite Exports in Capes,CHN Imports from Guinea,CHN Imports from Guinea inCapes
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,0.818666,-5.800000,7.300000,2833.218719,347.498651,243.250273,2357.945485,0.749897,0.585685,0.151858,...,0.176436,-0.347072,0.646973,0.109385,-308.266142,-495.915375,-1.460388,NaN,NaN,NaN
2013-01-02,0.641463,-6.487948,8.412519,2744.907012,362.292045,268.694052,2850.813524,0.803790,0.386925,-0.226175,...,0.160307,-0.347902,0.651900,0.178752,-307.513785,-533.209184,-1.427838,NaN,NaN,NaN
2013-01-03,0.556150,-5.392167,7.667353,2926.263479,395.185562,267.196156,2584.302096,0.727701,0.380782,-0.041591,...,0.296349,-0.348176,0.781480,0.162078,-304.596908,-552.897797,-1.402523,NaN,NaN,NaN
2013-01-04,0.822211,-4.147681,6.796589,3372.646099,432.950967,258.746924,2668.763126,0.717718,0.558667,0.344604,...,0.192700,-0.349004,0.684167,0.047224,-307.779353,-544.794490,-1.422653,NaN,NaN,NaN
2013-01-07,0.703521,-5.302108,6.868981,2858.098888,333.948645,240.636157,2113.238323,0.533633,0.475361,0.615358,...,0.179135,-0.347191,0.589435,0.102880,-421.108866,-527.529655,-1.493963,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.820397,-0.693937,-1.302623,-0.937099,-268.036101,-214.724667,NaN,NaN,NaN,NaN
2026-02-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.820940,-0.656495,-1.154943,-0.944037,-252.377347,-189.939481,NaN,NaN,NaN,NaN
2026-02-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.776112,-0.742490,-1.118786,-1.093195,-298.658380,-222.799734,NaN,NaN,NaN,NaN


In [12]:
features_des1 = transform_feature.agg(['mean', 'median', 'std', 'skew', 'kurtosis']).T
features_des1.columns = ["Mean", "Median", "STD", "Skewness", "Kurtosis"]
features_des1["Direction"] = "Normal"
features_des1.loc[features_des1["Skewness"] < -0.5, "Direction"] = "Left Skewed"
features_des1.loc[features_des1["Skewness"] > 0.5, "Direction"] = "Right Skewed"
features_des1

,Mean,Median,STD,Skewness,Kurtosis,Direction
Industrial Production China_% Yr/Yr,-1.012349e-16,-0.051536,1.000148,2.650991,26.155323,Right Skewed
Industrial Production Japan_% Yr/Yr,-1.439091e-01,-0.615459,5.836940,0.017834,5.193966,Normal
Industrial Production S Korea_% Yr/Yr,9.630932e-01,0.816101,4.867229,0.169272,0.563881,Normal
"China Steel Production_,000 tonnes",3.552380e+03,3501.364581,587.543951,0.393649,-0.225935,Normal
"Japan Steel Production_,000 tonnes",3.720117e+02,371.785009,57.571040,0.105250,-0.195519,Normal
...,...,...,...,...,...,...
steel margin cash profit with cost delayed 1 week for marginal producer of CRC,-7.302319e+01,-73.171187,482.727471,0.493599,0.348114,Normal
Guinea Total Bauxite Exports,-1.998011e-16,0.155313,1.000146,-0.066412,-1.153774,Normal
Guinea Bauxite Exports in Capes,5.851528e-16,-0.003534,1.000184,-0.124487,-0.619371,Normal
CHN Imports from Guinea,-1.483194e-16,0.067788,1.000163,-0.250464,-0.648413,Normal


In [13]:
features_des1.loc[features_des1["Direction"] == "Left Skewed"]

,Mean,Median,STD,Skewness,Kurtosis,Direction
"Japan BFI Production_,000 tonnes",-1.349799e-16,0.493103,1.000148,-0.863443,-1.127540,Left Skewed
"R.O.Korea BFI Production_,000 tonnes",-2.193424e-16,0.544636,1.000148,-0.896803,-1.113438,Left Skewed
Processing Cost,-4.131563e-16,0.695254,1.000151,-0.743407,-1.448224,Left Skewed


In [14]:
features_des1.loc[features_des1["Direction"] == "Right Skewed"]

,Mean,Median,STD,Skewness,Kurtosis,Direction
Industrial Production China_% Yr/Yr,-1.012349e-16,-0.051536,1.000148,2.650991,26.155323,Right Skewed
china new loan,1.340647e-16,-0.167961,1.000147,1.027149,3.036515,Right Skewed


After initial transformation, only 4 variable were skewed.

In [15]:
station_results_trans = []

for col in transform_feature.columns:
  series = transform_feature[col].dropna()
  if len(series) < 2 or series.nunique() < 2:
    continue
  try:
    result = adfuller(series)
    station = "Yes" if result[1] <= 0.05 else "No"
    station_results_trans.append({
        "Feature": col,
        "ADF_Statistic": round(result[0], 4),
        "p_value": round(result[1], 4),
        "Stationary": station
      })
  except:
    print(col)
    pass

_adf_cols = ["Feature", "ADF_Statistic", "p_value", "Stationary"]
station_results_df1 = pd.DataFrame(station_results_trans, columns=_adf_cols)
station_results_df1.loc[station_results_df1["Stationary"] == "No"]


,Feature,ADF_Statistic,p_value,Stationary
7,"Japan BFI Production_,000 tonnes",-1.7641,0.3984,No
8,"R.O.Korea BFI Production_,000 tonnes",-1.6516,0.4562,No
9,Japan Steel Ship Plate Commodity Price_$/Tonne,-1.2481,0.6526,No
10,Korea Steel Ship Plate Commodity Price_$/Tonne,-1.7876,0.3866,No
11,Iron Ore Spot Price CFR N China_$/Tonne,-2.1878,0.2107,No
...,...,...,...,...
174,steel margin cash profit with cost delayed 1 w...,-2.5278,0.1089,No
175,Guinea Total Bauxite Exports,-0.8143,0.8149,No
176,Guinea Bauxite Exports in Capes,-1.8419,0.3599,No
177,CHN Imports from Guinea,-1.0676,0.7278,No


After initial transformation, 109 of the 180 features remained non-stationary. We will now try to detrend and deseason the transformed dataset.

# Using FFT and ACF to identify the magnitude of seasonal and autoregressive factor


In [16]:
import numpy as np
from statsmodels.tsa.stattools import pacf


def get_optimal_lags_detrended(series, max_season=252, max_ar=40, min_len=40):
    """Seasonal period *m* from dominant FFT cycle; AR structure from PACF (lags with |PACF| > ~95% bound).

    Used by the next cell to build ``feature_seasonal_lag`` for ``strip_seasonality_and_ar``.
    Returns ``(m, list_of_candidate_ar_lags)``; callers use ``max(...)`` as the AutoReg order.
    """
    s = pd.Series(series).dropna().astype(float)
    n = len(s)
    if n < min_len:
        raise ValueError(f"Series too short ({n} < {min_len})")

    x = (s - s.mean()).to_numpy(dtype=float)
    spec = np.abs(np.fft.rfft(x)) ** 2
    best_m, best_v = 5, -1.0
    k_lo = max(1, int(np.ceil(n / max_season)))
    k_hi = min(len(spec) - 1, n // 2)
    for k in range(k_lo, k_hi + 1):
        period = n / k
        if period < 2 or period > max_season:
            continue
        if spec[k] > best_v:
            best_v = spec[k]
            best_m = int(round(period))

    best_m = max(2, min(int(best_m), max_season))
    if best_v <= 0:
        best_m = 5

    nlags = min(max(1, max_ar), max(1, n // 4 - 1))
    pa = pacf(x, nlags=nlags, method="ywm")
    crit = 1.96 / np.sqrt(n)
    ar_lags = [i for i in range(1, min(nlags + 1, len(pa))) if abs(pa[i]) > crit]
    if not ar_lags:
        j = 1 if len(pa) <= 2 else int(np.argmax(np.abs(pa[1:])) + 1)
        ar_lags = [max(1, min(j, nlags))]

    return best_m, ar_lags


In [17]:
feature_seasonal_lag = {}

for column in transform_feature.columns:
    try:
        optimal_seasonal_lag, optimal_ar_lags = get_optimal_lags_detrended(transform_feature[column])

        max_ar = max(optimal_ar_lags) if optimal_ar_lags else 0

        feature_seasonal_lag[column] = {
            "Seasonal Lag": optimal_seasonal_lag,
            "Autoregressive Lags": max_ar
        }

    except Exception as e:
        print(f"Skipping '{column}' due to error: {e}")

Skipping 'Wire Rod 6.5 mm' due to error: Series too short (0 < 40)


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1490: ValueWarning: Matrix is singular. Using pinv.
  warnings.warn("Matrix is singular. Using pinv.", ValueWarning)


Skipping 'coking coal Australia low-vol coking coal (USD/t) - RHS USD/t FOB' due to error: Series too short (13 < 40)
Skipping 'coking coal USD/t FOB' due to error: Series too short (13 < 40)


In [18]:
feature_seasonal_lag = pd.DataFrame(feature_seasonal_lag).T
feature_seasonal_lag

,Seasonal Lag,Autoregressive Lags
Industrial Production China_% Yr/Yr,187,33
Industrial Production Japan_% Yr/Yr,167,25
Industrial Production S Korea_% Yr/Yr,49,25
"China Steel Production_,000 tonnes",62,35
"Japan Steel Production_,000 tonnes",62,39
...,...,...
steel margin cash profit with cost delayed 1 week for marginal producer of CRC,132,39
Guinea Total Bauxite Exports,244,27
Guinea Bauxite Exports in Capes,247,36
CHN Imports from Guinea,219,23


# Removing seasonality and autoregression based on the FFT and ACF analysis.

In [19]:
import pandas as pd
from statsmodels.tsa.ar_model import AutoReg

def strip_seasonality_and_ar(df, lag_info):
    stripped_df = pd.DataFrame(index=df.index)

    for column in df.columns:
        if column not in df.columns:
            continue

        try:
            if isinstance(lag_info, pd.DataFrame):
                if 'Seasonal Lag' in lag_info.columns:
                    m = int(lag_info.loc[column, 'Seasonal Lag'])
                    p = int(lag_info.loc[column, 'Autoregressive Lags'])
                else:
                    m = int(lag_info.loc['Seasonal Lag', column])
                    p = int(lag_info.loc['Autoregressive Lags', column])
            else:
                m = int(lag_info[column]["Seasonal Lag"])
                p = int(lag_info[column]["Autoregressive Lags"])
        except KeyError:
            print(f"Skipping {column}: Could not find lag parameters.")
            continue

        series = df[column].dropna()
        if len(series) == 0:
            continue

        if m > 0:
            deseasonalized = series.diff(m).dropna()
        else:
            deseasonalized = series.copy()

        if p > 0 and len(deseasonalized) > p:
            model = AutoReg(deseasonalized, lags=p, old_names=False).fit()
            stripped_series = model.resid
        else:
            stripped_series = deseasonalized

        # Column name includes season length (days) and AR lag for transparency
        out_col = f"{column}_season{m}d_ar{p}"
        stripped_df[out_col] = stripped_series

    return stripped_df

df_residuals = strip_seasonality_and_ar(transform_feature, feature_seasonal_lag)
df_residuals = pd.DataFrame(df_residuals)
df_residuals

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was pr

Skipping Wire Rod 6.5 mm: Could not find lag parameters.


/var/folders/ym/q8pn40b973g61znsnb_4mfvc0000gn/T/ipykernel_23035/1135711798.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stripped_df[out_col] = stripped_series
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/ym/q8pn40b973g61znsnb_4mfvc0000gn/T/ipykernel_23035/1135711798.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe =

Skipping coking coal Australia low-vol coking coal (USD/t) - RHS USD/t FOB: Could not find lag parameters.
Skipping coking coal USD/t FOB: Could not find lag parameters.


/var/folders/ym/q8pn40b973g61znsnb_4mfvc0000gn/T/ipykernel_23035/1135711798.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stripped_df[out_col] = stripped_series
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/var/folders/ym/q8pn40b973g61znsnb_4mfvc0000gn/T/ipykernel_23035/1135711798.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  stripped_df[out_col] = stripped

,Industrial Production China_% Yr/Yr_season187d_ar33,Industrial Production Japan_% Yr/Yr_season167d_ar25,Industrial Production S Korea_% Yr/Yr_season49d_ar25,"China Steel Production_,000 tonnes_season62d_ar35","Japan Steel Production_,000 tonnes_season62d_ar39","South Korea Steel Production_,000 tonnes_season62d_ar40","P.R. China BFI Production_,000 tonnes_season62d_ar29","Japan BFI Production_,000 tonnes_season211d_ar23","R.O.Korea BFI Production_,000 tonnes_season211d_ar22",Japan Steel Ship Plate Commodity Price_$/Tonne_season188d_ar24,...,steel margin cash profit with cost delayed 1 month for marginal producer of Rebar_season214d_ar25,steel margin cash profit with cost delayed 1 week for marginal producer of Rebar_season214d_ar30,steel margin cash profit with cost delayed 1 month for marginal producer of HRC_season156d_ar20,steel margin cash profit with cost delayed 1 week for marginal producer of HRC_season214d_ar37,steel margin cash profit with cost delayed 1 month for marginal producer of CRC_season245d_ar40,steel margin cash profit with cost delayed 1 week for marginal producer of CRC_season132d_ar39,Guinea Total Bauxite Exports_season244d_ar27,Guinea Bauxite Exports in Capes_season247d_ar36,CHN Imports from Guinea_season219d_ar23,CHN Imports from Guinea inCapes_season247d_ar6
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.088578,0.102240,-0.170455,0.146711,-4.409401,-4.999964,NaN,NaN,NaN,NaN
2026-02-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000320,0.073254,0.136846,0.054242,6.984265,23.739788,NaN,NaN,NaN,NaN
2026-02-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.016668,-0.129951,0.109450,-0.146702,-36.793977,-21.160639,NaN,NaN,NaN,NaN


In [20]:
station_results_trans = []

for col in df_residuals.columns:
  series = df_residuals[col].dropna()
  if len(series) < 2 or series.nunique() < 2:
    continue
  try:
    result = adfuller(series)
    station = "Yes" if result[1] <= 0.05 else "No"
    station_results_trans.append({
        "Feature": col,
        "ADF_Statistic": round(result[0], 4),
        "p_value": round(result[1], 4),
        "Stationary": station
      })
  except:
    print(col)
    pass

_adf_cols = ["Feature", "ADF_Statistic", "p_value", "Stationary"]
station_results_df3 = pd.DataFrame(station_results_trans, columns=_adf_cols)
station_results_df3.loc[station_results_df3["Stationary"] == "No"]


,Feature,ADF_Statistic,p_value,Stationary


We have identify 1 independent variable that remains non-stationary

# Part 2 — First-order differencing (trend / seasonality)

Same **`transform_feature`** / **`features`** sample as Part 1 (through **`DATA_END`** = last row in master, including **2025** when available). Final save step writes **`Transformed_Deseason_Diff1__from_<INPUT_STEM>.xlsx`** (stem from the master file loaded at the top).


In [21]:
diff_1_trans = transform_feature.diff()

In [22]:
station_results_trans = []

for col in diff_1_trans.columns:
  series = diff_1_trans[col].dropna()
  if len(series) < 2 or series.nunique() < 2:
    continue
  try:
    result = adfuller(series)
    station = "Yes" if result[1] <= 0.05 else "No"
    station_results_trans.append({
        "Feature": col,
        "ADF_Statistic": round(result[0], 4),
        "p_value": round(result[1], 4),
        "Stationary": station
      })
  except:
    print(col)
    pass

_adf_cols = ["Feature", "ADF_Statistic", "p_value", "Stationary"]
station_results_df2 = pd.DataFrame(station_results_trans, columns=_adf_cols)
station_results_df2.loc[station_results_df2["Stationary"] == "No"]


,Feature,ADF_Statistic,p_value,Stationary


Similar results as the FFT and ACF method, same variable remained non-stationary

In [23]:
# Save both result versions (filenames include INPUT_STEM from first cell)
output_dir = BASE_DIR
_tag = f"__from_{INPUT_STEM}"
_res_path = os.path.join(output_dir, f"Transformed_Deseason_Residuals{_tag}.xlsx")
_diff_path = os.path.join(output_dir, f"Transformed_Deseason_Diff1{_tag}.xlsx")
df_residuals.to_excel(_res_path, index=True)
diff_1_trans.to_excel(_diff_path, index=True)
print(f"Saved (FFT/ACF): {_res_path}")
print(f"Saved (first-diff): {_diff_path}")

Saved (FFT/ACF): /Users/andyyang/Downloads/Project 005 Data/Transformed_Deseason_Residuals__from_Master_Data_Base_version_Brownian_by_frequency.xlsx
Saved (first-diff): /Users/andyyang/Downloads/Project 005 Data/Transformed_Deseason_Diff1__from_Master_Data_Base_version_Brownian_by_frequency.xlsx
